In [ ]:
import pandas as pd
import numpy as np
from sklearn.ensemble import RandomForestClassifier, StackingClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import StratifiedKFold, cross_val_score
from xgboost import XGBClassifier
import warnings
warnings.filterwarnings('ignore')

print("Cell 1: Library berhasil dimuat!")

In [ ]:
# Membaca dataset
train_df = pd.read_csv('train.csv')
test_df = pd.read_csv('test.csv')

# Menyimpan ID untuk file submission
test_ids = test_df['PassengerId']

print(f"Cell 2: Data Train ({train_df.shape[0]} baris), Data Test ({test_df.shape[0]} baris) siap!")

In [ ]:
# Gabungkan data untuk pemrosesan konsisten
full_df = pd.concat([train_df, test_df], sort=False).reset_index(drop=True)

# 1. Ekstraksi Gelar (Title)
full_df['Title'] = full_df['Name'].str.extract(' ([A-Za-z]+)\.', expand=False)
title_mapping = {'Mr': 'Mr', 'Miss': 'Miss', 'Mrs': 'Mrs', 'Master': 'Master'}
full_df['Title'] = full_df['Title'].map(title_mapping).fillna('Rare')

# 2. Ukuran Keluarga
full_df['FamilySize'] = full_df['SibSp'] + full_df['Parch'] + 1

# 3. Analisis Prefiks Tiket (Agen Perjalanan/Jalur Evakuasi)
full_df['Ticket_Prefix'] = full_df['Ticket'].apply(lambda x: x.split()[0].replace('.', '').replace('/', '') if not x.isdigit() else 'None')

# 4. [FITUR EMAS] Group Survival Rate
full_df['Surname'] = full_df['Name'].apply(lambda x: x.split(',')[0].strip())
full_df['Group_Survival'] = 0.5 # Nilai tengah (belum diketahui)

# Lacak berdasarkan Nama Belakang dan Tarif
for grp, grp_df in full_df.groupby(['Surname', 'Fare']):
    if len(grp_df) > 1:
        for ind, row in grp_df.iterrows():
            smax = grp_df.drop(ind)['Survived'].max()
            smin = grp_df.drop(ind)['Survived'].min()
            passID = row['PassengerId']
            if smax == 1.0:
                full_df.loc[full_df['PassengerId'] == passID, 'Group_Survival'] = 1
            elif smin == 0.0:
                full_df.loc[full_df['PassengerId'] == passID, 'Group_Survival'] = 0

# Lacak berdasarkan Nomor Tiket (untuk teman/rombongan)
for grp, grp_df in full_df.groupby('Ticket'):
    if len(grp_df) > 1:
        for ind, row in grp_df.iterrows():
            if full_df.loc[full_df['PassengerId'] == row['PassengerId'], 'Group_Survival'].values[0] == 0.5:
                smax = grp_df.drop(ind)['Survived'].max()
                smin = grp_df.drop(ind)['Survived'].min()
                passID = row['PassengerId']
                if smax == 1.0:
                    full_df.loc[full_df['PassengerId'] == passID, 'Group_Survival'] = 1
                elif smin == 0.0:
                    full_df.loc[full_df['PassengerId'] == passID, 'Group_Survival'] = 0

print("Cell 3: Rekayasa Fitur Emas selesai!")

In [ ]:
# 1. Imputasi berbasis kelompok
full_df['Age'] = full_df.groupby(['Title', 'Pclass'])['Age'].transform(lambda x: x.fillna(x.median()))
full_df['Fare'] = full_df.groupby('Pclass')['Fare'].transform(lambda x: x.fillna(x.median()))
full_df['Embarked'] = full_df['Embarked'].fillna('S')

# 2. Imputasi Sapu Jagat (menutup sisa NaN yang bocor)
full_df['Age'] = full_df['Age'].fillna(full_df['Age'].median())
full_df['Fare'] = full_df['Fare'].fillna(full_df['Fare'].median())

# 3. Hapus kolom mentah yang tidak diperlukan lagi
drop_cols = ['PassengerId', 'Survived', 'Name', 'Ticket', 'Cabin', 'Surname', 'Ticket_Prefix']

# 4. One-Hot Encoding
categorical_cols = ['Sex', 'Embarked', 'Title']
cols_to_dummy = [col for col in categorical_cols if col in full_df.columns]
if cols_to_dummy:
    full_df = pd.get_dummies(full_df, columns=cols_to_dummy, drop_first=True)

# 5. Konversi tipe boolean ke integer agar aman dibaca model
cols_to_convert = [c for c in full_df.columns if full_df[c].dtype == 'bool']
if cols_to_convert:
    full_df[cols_to_convert] = full_df[cols_to_convert].astype(int)

# 6. Pemisahan Data & Pembuatan Matriks X dan y
train_clean = full_df[full_df['Survived'].notnull()].copy()
test_clean = full_df[full_df['Survived'].isnull()].copy()

X = train_clean.drop(columns=drop_cols, errors='ignore')
y = train_clean['Survived'].astype(int)
X_test = test_clean.drop(columns=drop_cols, errors='ignore')

# 7. Penyelarasan kolom dan perlindungan akhir dari NaN
X, X_test = X.align(X_test, join='left', axis=1, fill_value=0)
X = X.fillna(0)
X_test = X_test.fillna(0)

print(f"Cell 4: Matriks X siap dengan {X.shape[1]} fitur!")

In [ ]:
# 1. Definisikan Model Dasar (Base Models)
base_models = [
    ('rf', RandomForestClassifier(n_estimators=100, max_depth=5, random_state=42)),
    ('xgb', XGBClassifier(n_estimators=100, max_depth=4, learning_rate=0.05, random_state=42, eval_metric='logloss'))
]

# 2. Definisikan Model Meta (Bos yang belajar dari model dasar)
meta_model = LogisticRegression(random_state=42)

# 3. Bangun Arsitektur Stacking
stacking_clf = StackingClassifier(estimators=base_models, final_estimator=meta_model, cv=5)

# 4. Evaluasi Performa Model Meta
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
scores = cross_val_score(stacking_clf, X, y, cv=cv, scoring='accuracy')

print(f"Akurasi Stacking di Data Train: {scores.mean() * 100:.2f}%")

# 5. Latih model secara keseluruhan
stacking_clf.fit(X, y)
print("Cell 5: Model Stacking berhasil dilatih!")

In [ ]:
# 1. Prediksi menggunakan Stacking Classifier
predictions = stacking_clf.predict(X_test)

# 2. Kemas menjadi format kompetisi
submission = pd.DataFrame({
    'PassengerId': test_ids,
    'Survived': predictions
})

# 3. Simpan menjadi CSV (pastikan tidak ada kolom ekstra/indeks)
submission.to_csv('submission_titanic_final.csv', index=False)

print("Cell 6: File 'submission_titanic_final.csv' berhasil digenerate dan siap di-submit ke Kaggle!")